In [1]:
import os
import math
import random
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

from src.data import get_divided_loaders
from src.models import TransformerAutoencoder

In [2]:
# def train_epoch(model, loader, optimizer, criterion, device, epoch_num=''):
#     model.train()
#     total, count = 0.0, 0
#     pbar = tqdm(loader, total=len(loader), desc=f'Epoch {epoch_num}', leave=True)
#     for src in pbar:
#         src = src.to(device)

#         optimizer.zero_grad()
#         logits = model(src)
#         loss = criterion(logits.view(-1, logits.size(-1)), src.view(-1))
#         loss.backward()
#         optimizer.step()

#         total += loss.item() * src.size(0)
#         count += src.size(0)
#     return total / count

def train_epoch(model, dataloader, optimizer, device, epoch_num='', mse_weight=1.0, clip_grad=None):
    model.train()
    running_loss = 0.0
    criterion = nn.MSELoss(reduction='mean')

    pbar = tqdm(dataloader, desc=f'Train epoch {epoch_num}', leave=True)
    for batch in pbar:
        x = batch.to(device)
        optimizer.zero_grad()
        recon = model(x)
        loss = criterion(recon, x) * mse_weight
        loss.backward()
        if clip_grad is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        optimizer.step()
        running_loss += loss.item() * x.size(0)

        pbar.set_postfix(train_loss=running_loss / ((pbar.n + 1) * dataloader.batch_size))
        torch.mps.empty_cache()
    return running_loss / len(dataloader.dataset)


# def evaluate(model, loader, criterion, device):
#     model.eval()
#     total, count = 0.0, 0
#     with torch.no_grad():
#         for src in loader:
#             src = src.to(device)
#             logits = model(src)
#             loss = criterion(logits.view(-1, logits.size(-1)), src.view(-1))
#             total += loss.item() * src.size(0)
#             count += src.size(0)
#     return total / count

def evaluate(model, dataloader, device='mps', epoch_num=''):
    model.eval()
    criterion = nn.MSELoss(reduction='mean')
    all_losses = []
    running_loss = 0.0
    torch.mps.empty_cache()

    pbar = tqdm(dataloader, desc=f"Validate epoch {epoch_num}", leave=True)
    with torch.no_grad():
        for batch in pbar:
            x = batch.to(device)
            recon = model(x)
            recon_cpu = recon.detach().to("cpu")
            del x, recon
            
            # per_elem = criterion(recon_cpu, batch)  # (batch, seq_len, 1)
            # per_sample_mse = per_elem.mean(dim=(1, 2)).cpu()  # (batch,)
            # # all_losses.append(per_sample_mse)
            # running_loss += per_sample_mse.mean().item()

            loss = criterion(recon_cpu, batch)
            running_loss += loss.item() * batch.size(0)

            pbar.set_postfix(train_loss=running_loss / ((pbar.n + 1) * dataloader.batch_size))
            

            torch.mps.empty_cache()
            
    # all_losses = torch.cat(all_losses, dim=0)
    # return all_losses
    return running_loss / len(dataloader.dataset)


In [3]:

def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)

# -------------------------
# Optuna Objective
# -------------------------
def objective(trial):
    set_seed(42)

    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

    # Hyperparameters
    # d_model = trial.suggest_categorical("d_model", [128, 256, 384])
    # nhead = trial.suggest_categorical("nhead", [2, 4, 8])
    # num_layers = trial.suggest_int("num_layers", 1, 4)
    # dim_ff = trial.suggest_int("dim_feedforward", 256, 1024, step=128)

    d_model = 64
    nhead = 4
    num_layers = 2
    dim_ff = 2048

    # dropout = trial.suggest_float("dropout", 0.0, 0.3)
    dropout = 0.1 # MORE OPEN TO OPTIMIZING THIS

    # lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
    # weight_decay = trial.suggest_loguniform("weight_decay", 1e-8, 1e-3)
    # batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    lr = 1e-3
    weight_decay = 0.01
    batch_size = 64

    window_skip = trial.suggest_int("window_skip", 300, 800, step=100)
    window_size = trial.suggest_int("window_size", 500, 1500, step=500)
    train_loader, val_loader = get_divided_loaders(
        test_size=0.5,
        window_skip=window_skip, 
        window_size=window_size,
        preload=True
    )

    transformer_model = TransformerAutoencoder(
        input_dim=1,      # number of features per timestep
        model_dim=64,     # hidden embedding dimension
        num_heads=4,      # parallel attention heads
        num_layers=2,      # stacked encoder/decoder layers
        # dropout=dropout
    ).to(device)

    optimizer = optim.AdamW(transformer_model.parameters(), lr=lr)
    # criterion = nn.CrossEntropyLoss()

    max_epochs = 3
    best_val = float("inf")

    for epoch in range(1,1+max_epochs):
        train_loss = train_epoch(transformer_model, 
                                 dataloader=train_loader, 
                                 optimizer=optimizer, 
                                 device=device, 
                                 epoch_num=epoch,
                                 clip_grad=1.0)
        val_loss = evaluate(transformer_model, dataloader=val_loader, device=device)

        # Report intermediate result
        trial.report(val_loss, epoch)

        # Pruning
        if trial.should_prune():
            raise optuna.TrialPruned()

        best_val = min(best_val, val_loss)

    return best_val


In [16]:
train_loader, val_loader = get_divided_loaders(
        test_size=0.1,
        window_skip=500,
        preload=True
    )

DEVICE = 'mps' if torch.mps.is_available() else 'cpu'

transformer_model = TransformerAutoencoder(
    input_dim=1,      # number of features per timestep
    model_dim=64,     # hidden embedding dimension
    num_heads=4,      # parallel attention heads
    num_layers=2,      # stacked encoder/decoder layers
    # dropout=dropout
).to(DEVICE)

optimizer = optim.AdamW(transformer_model.parameters(), lr=1e-3)

# train_epoch(transformer_model, dataloader=train_loader, optimizer=optimizer, device='mps', epoch_num=1)
eval = evaluate(transformer_model, val_loader, DEVICE)
eval

Checking attributes: 100%|██████████| 333/333 [00:00<00:00, 3490.82it/s]


Building dataset index...


100%|██████████| 93/93 [00:00<00:00, 906.96it/s]


Building dataset index...


100%|██████████| 10/10 [00:00<00:00, 1006.43it/s]
/Users/Hanita/Projects/Environments/ml_venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:393: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Validate: 100%|██████████| 63/63 [00:51<00:00,  1.21it/s]


831.9536828372012

In [4]:
study = optuna.create_study(
    study_name="transformer_mps_opt",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=4),
)

study.optimize(objective, n_trials=3)

print("Best trial:", study.best_trial.value)
print("Best params:", study.best_trial.params)

[I 2025-11-18 11:16:42,567] A new study created in memory with name: transformer_mps_opt
Checking attributes: 100%|██████████| 333/333 [00:00<00:00, 1093.22it/s]


Building dataset index...


100%|██████████| 52/52 [00:00<00:00, 973.35it/s]


Building dataset index...


100%|██████████| 51/51 [00:00<00:00, 948.69it/s]
/Users/Hanita/Projects/Environments/ml_venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:393: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Validate: 100%|██████████| 384/384 [54:57<00:00,  8.59s/it]                     
[I 2025-11-18 17:58:09,623] Trial 0 finished with value: 36.50720292708092 and parameters: {'window_skip': 500, 'window_size': 1500}. Best is trial 0 with value: 36.50720292708092.
Checking attributes: 100%|██████████| 333/333 [00:00<00:00, 947.62it/s] 


...

Building dataset index...


100%|██████████| 52/52 [00:00<00:00, 1034.73it/s]


Building dataset index...


Validate: 100%|██████████| 641/641 [24:48<00:00,  2.32s/it]                    
[I 2025-11-19 00:07:55,681] Trial 2 finished with value: 15.035340816302107 and parameters: {'window_skip': 300, 'window_size': 500}. Best is trial 2 with value: 15.035340816302107.


Best trial: 15.035340816302107
Best params: {'window_skip': 300, 'window_size': 500}
